1- Analisis Descriptivo y estadístrico básico

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("../data/spotify_procesado01.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 113549 entries, 0 to 113548
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          113549 non-null  str    
 1   artists           113549 non-null  str    
 2   album_name        113549 non-null  str    
 3   track_name        113549 non-null  str    
 4   popularity        113549 non-null  int64  
 5   duration_ms       113549 non-null  int64  
 6   explicit          113549 non-null  bool   
 7   danceability      113549 non-null  float64
 8   energy            113549 non-null  float64
 9   key               113549 non-null  int64  
 10  loudness          113549 non-null  float64
 11  mode              113549 non-null  int64  
 12  speechiness       113549 non-null  float64
 13  acousticness      113549 non-null  float64
 14  instrumentalness  113549 non-null  float64
 15  liveness          113549 non-null  float64
 16  valence           113549 non-nu

In [ ]:


print(df[["popularity", "energy", "valence", "danceability"]].describe())

          popularity         energy        valence   danceability
count  113549.000000  113549.000000  113549.000000  113549.000000
mean       33.324433       0.642091       0.474205       0.567031
std        22.283855       0.251053       0.259204       0.173409
min         0.000000       0.000000       0.000000       0.000000
25%        17.000000       0.473000       0.260000       0.456000
50%        35.000000       0.685000       0.464000       0.580000
75%        50.000000       0.854000       0.683000       0.695000
max       100.000000       1.000000       0.995000       0.985000


Valence media (0.47): Queda por debajo de la mitad de la escala, lo que indica que, en promedio, el catálogo global ya tiende hacia tonalidades más melancólicas, oscuras o tensas en lugar de puramente alegres.

Energy media (0.64): Es notablemente alta, lo que confirma que predomina un estímulo físico dinámico e intenso.

2- Aislar la musica urbana: identificar generos, 

In [18]:

print(df["track_genre"].unique()[:])

<StringArray>
[   'acoustic',    'afrobeat',    'alt-rock', 'alternative',     'ambient',
       'anime', 'black-metal',   'bluegrass',       'blues',      'brazil',
 ...
     'spanish',       'study',     'swedish',   'synth-pop',       'tango',
      'techno',      'trance',    'trip-hop',     'turkish', 'world-music']
Length: 114, dtype: str


2.1.- Generos más populares

In [12]:
resumen_generos = df.groupby("track_genre").agg({
    "popularity": "mean",
    "energy": "mean",
    "valence": "mean",
    "explicit": "mean",
    "track_id": "count"
}).rename(columns={"track_id": "num_tracks"}).reset_index()

print(resumen_generos.sort_values(by="popularity", ascending=False).head(20))

          track_genre  popularity    energy   valence  explicit  num_tracks
81           pop-film   59.280280  0.604385  0.528981  0.001001         999
65              k-pop   56.963928  0.675851  0.556504  0.049098         998
15              chill   53.704705  0.426399  0.403545  0.171171         999
94                sad   52.379000  0.462470  0.422050  0.450000        1000
44             grunge   49.582583  0.803112  0.400448  0.072072         999
55             indian   49.528529  0.567138  0.463315  0.024024         999
5               anime   48.766767  0.673788  0.434636  0.055055         999
33                emo   48.128000  0.669967  0.441135  0.465000        1000
80                pop   47.903323  0.605502  0.506931  0.072508         993
97          sertanejo   47.866000  0.710391  0.619325  0.006000        1000
83  progressive-house   46.661662  0.813243  0.365326  0.052052         999
79              piano   45.277555  0.320601  0.313714  0.012024         998
70          

In [14]:
resumen_generos = df.groupby("track_genre").agg({
    "popularity": "mean",
    "energy": "mean",
    "valence": "mean",
    "explicit": "mean",
    "track_id": "count"
}).rename(columns={"track_id": "num_tracks"}).reset_index()

urbano = resumen_generos["track_genre"].str.contains("reggaeton|latin|urban|hip-hop|trap", case=False, na=False)
print(resumen_generos[urbano].sort_values(by="popularity", ascending=False))

   track_genre  popularity    energy   valence  explicit  num_tracks
51     hip-hop   38.082745  0.682361  0.550513  0.316852         991
68      latino   25.826788  0.730758  0.628753  0.249748         993
89   reggaeton   23.858000  0.738728  0.642753  0.212000        1000
67       latin    8.363636  0.727144  0.632097  0.132323         990


correlación entre las variables

In [15]:
print(df[["popularity", "energy", "valence", "danceability", "explicit"]].corr())

              popularity    energy   valence  danceability  explicit
popularity      1.000000 -0.002447 -0.041097      0.034407  0.043938
energy         -0.002447  1.000000  0.258451      0.131694  0.096256
valence        -0.041097  0.258451  1.000000      0.476755 -0.003779
danceability    0.034407  0.131694  0.476755      1.000000  0.122179
explicit        0.043938  0.096256 -0.003779      0.122179  1.000000


Conclusión: no hay relación lineal relevante entre "popularity" y el resto de variables por lo que el exito de las canciones atiende a otras razones. 

2.2- aislar los generos de alta rotación: 1º filtro canciones con popularidad superior a 50. (Hipotesis uno)

In [17]:
hits = df[df["popularity"] > 50]

print(hits[["energy", "valence", "danceability", "explicit"]].mean())

energy          0.631438
valence         0.455659
danceability    0.582738
explicit        0.114094
dtype: float64


- La valencia baja aún más (pasa de 0.474 del total de todo el Dataset a 0.455), lo que refuerza empíricamente la Hipótesis 1: los temas que alcanzan mayor relevancia comercial tienden a concentrarse en zonas de mayor tensión emocional o melancolía.

- La bailabilidad y la presencia de contenido explícito suben (la proporción de canciones explícitas pasa a ser del 11.4% en los éxitos, un dato muy revelador sobre el perfil del consumo masivo).

2.3- Que generos dominan los éxitos

In [22]:
hits = df[df["popularity"] > 50]

print(hits["track_genre"].value_counts().head(15))

track_genre
pop-film             942
chill                759
k-pop                684
sad                  654
pop                  644
grunge               617
hip-hop              572
progressive-house    571
metal                561
british              542
deep-house           527
psych-rock           505
emo                  501
electro              499
metalcore            497
Name: count, dtype: int64


Conclusion : hip-hop se cuela directamente en el top de éxitos comerciales junto con pop, k-pop y géneros de alta carga emocional como sad o grunge. Esto da una base empírica sólida para argumentar que el consumo masivo no está dominado por temas neutros o hiper-alegres, sino por categorías que oscilan entre el gancho comercial (pop) y la narrativa de conflicto/tensión (hip-hop, sad, grunge).

In [27]:
df["pop_tier"] = pd.qcut(df["popularity"], q=4, labels=["Q1 (Baja)", "Q2", "Q3", "Q4 (Alta)"])
print(df.groupby("pop_tier")[["energy", "valence", "danceability", "explicit"]].mean())

             energy   valence  danceability  explicit
pop_tier                                             
Q1 (Baja)  0.637521  0.492261      0.581409  0.073556
Q2         0.660074  0.477774      0.533629  0.087560
Q3         0.638430  0.470318      0.571825  0.067414
Q4 (Alta)  0.631438  0.455659      0.582738  0.114094


In [30]:
urbano = df["track_genre"].str.contains("hip-hop|rap|reggaeton|latin|trap", case=False, na=False)

print("--- Perfil Géneros Urbanos/Comerciales ---")
print(df[urbano][["energy", "valence", "danceability", "explicit", "popularity"]].mean())

print("\n--- Perfil Resto del Dataset ---")
print(df[~urbano][["energy", "valence", "danceability", "explicit", "popularity"]].mean())

--- Perfil Géneros Urbanos/Comerciales ---
energy           0.719795
valence          0.613598
danceability     0.743363
explicit         0.227730
popularity      24.037242
dtype: float64

--- Perfil Resto del Dataset ---
energy           0.639273
valence          0.469150
danceability     0.560636
explicit         0.080429
popularity      33.661255
dtype: float64
